# Lesson 04 - टुल प्रयोग डिजाइन ढाँचा

यस पाठमा तपाईं Microsoft Agent Framework (Python) प्रयोग गरेर AI एजेन्टहरूको लागि **टुल प्रयोग** डिजाइन ढाँचा सिक्नुहुनेछ। हामीले समावेश गरेका छौं:

- `@tool` डेकोरेटर र टाइप गरिएको प्यारामिटरहरूसँग 함수 टुलहरू परिभाषित गर्ने
- मोडेललाई प्रत्येक टुलले के गर्छ भनेर थाहा पाउन टुल स्कीमाहरू प्रदान गर्ने
- `approval_mode` द्वारा टुल कार्यान्वयन नियन्त्रण गर्ने
- Pydantic मोडेलहरू र `response_format` मार्फत **संरचित नतिजा** फर्काउने

परिदृश्य एक **यात्रा बुकिंग एजेन्ट** हो जसले गन्तव्यहरु खोज्न, उपलब्धता जाँच गर्न, र उडान सूचना प्राप्त गर्न सक्छ।


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## @tool डेकोरेटरका साथ उपकरणहरू परिभाषित गर्दै

`@tool` डेकोरेटरले साधारण Python फङ्क्शनलाई यस्तो उपकरणमा परिवर्तन गर्छ जुन एजेन्टले कल गर्न सक्छ।  
महत्त्वपूर्ण बुँदाहरू:

- **डकस्ट्रिङ** उपकरणको वर्णन हुन्छ जुन मोडलले हेर्छ।  
- **प्रकार एनोटेसनहरू** (वर्णनहरूसहित `Annotated` समेत) उपकरण स्किमालाई परिभाषित गर्छ।  
- `approval_mode` ले नियन्त्रण गर्छ कि कार्यान्वयन हुनु अघि प्रयोगकर्ताले हरेक कललाई अनुमोदन गर्नुपर्छ कि पर्दैन।


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## धेरै उपकरणहरूसँग एजेन्ट बनाउँदै

तीनै उपकरणहरूलाई क्लाइन्टमा पठाउनुहोस् ताकि मोडेलले प्रयोगकर्ताको प्रश्नको उत्तर दिन आवश्यक पर्ने जुनसुकै उपकरणलाई बोलाउन सकून्।


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## उपकरणहरूसँग संरचित आउटपुट

`response_format` लाई Pydantic मोडेलमा सेट गर्दा, एजेन्टलाई स्वतन्त्र पाठको सट्टा राम्रोसँग प्रकारबद्ध JSON वस्तु फर्काउन बाध्य पारिन्छ। जब डाउनस्ट्रीम कोडले परिणामलाई प्रोग्राममैटिक रूपमा प्रयोग गर्न आवश्यक पर्दछ, यो उपयोगी हुन्छ।


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## उपकरण स्वीकृति ढाँचाहरू

`@tool` मा रहेको `approval_mode` प्यारामीटरले उपकरण कलहरू क्रियान्वयन गर्नु अघि मानवीय स्वीकृति आवश्यक छ कि छैन भनेर नियन्त्रण गर्दछ:

| मोड | व्यवहार |
|---|---|
| `"never_require"` | उपकरण स्वतः चल्छ — प्रयोगकर्ता पुष्टि आवश्यक छैन। |
| `"always_require"` | प्रत्येक कललाई क्रियान्वयन हुनु अघि प्रयोगकर्ताद्वारा स्वीकृत हुनुपर्छ। |

एउटा मान्छे प्रक्रियामा रहोस् भनेर असर पर्ने उपकरणहरू (जस्तै उडान बुकिङ, क्रेडिट कार्ड चार्ज गर्नु) को लागि `"always_require"` प्रयोग गर्नुहोस्।


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## सारांश

यस पाठमा तपाईंले सिक्नुभयो कसरी:

1. **उपकरणहरू परिभाषित गर्ने** `@tool` डेकोरेटर प्रयोग गरेर टाइप गरिएको प्यारामिटरहरू र टुल स्कीमा को रूपमा सेवा गर्ने डक्स्ट्रिङसँग।
2. **धेरै उपकरणहरू संयोजन गर्ने** ताकि एजेन्टले जटिल प्रश्नहरूको उत्तर दिनको लागि तिनीहरूलाई अनुक्रममा कल गर्न सकोस्।
3. **संरचित आउटपुट फर्काउने** Pydantic मोडेललाई `response_format` को रूपमा पास गरेर।
4. **उपकरण स्वीकृतिलाई नियन्त्रण गर्ने** `approval_mode` सँग संवेदनशील कार्यहरूको लागि मानवलाई लूपमा राख्न।

यी ढाँचा भरपर्दो, उत्पादन-तयार एजेन्टहरू निर्माण गर्नको लागि आधार बनाउँछन् जसले बाह्य सिस्टमहरूसँग सुरक्षित रूपमा अन्तरक्रिया गर्न सक्छन्।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
